### Demo - Functions as Tool

#### 🔧 What Are Functions as Tools(Custom Tools) in Agents?
* Functions as tools allow you to wrap regular Python functions using the @function_tool decorator so that agents can call those functions autonomously when needed during a conversation or task.

* Think of them as custom plugins you create for your agent — the agent understands what the function does and uses it like a tool whenever appropriate.

In [9]:
# Necessary imports
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
# API Calls
import requests

In [8]:

load_dotenv()

True

#### GPA Agent with Function Tools

In [3]:
# GPA conversion helper based on your grading scale
def get_grade_point(percent: float) -> float:
    if 96 <= percent <= 100:
        return 4.33
    elif 91 <= percent < 96:
        return 4.0
    elif 86 <= percent < 91:
        return 3.67
    elif 81 <= percent < 86:
        return 3.33
    elif 76 <= percent < 81:
        return 3.0
    elif 71 <= percent < 76:
        return 2.67
    elif 66 <= percent < 71:
        return 2.33
    elif 60 <= percent < 66:
        return 2.0
    else:
        return 0.0  # Grade No Credits

In [8]:
# Wrap the GPA calculator as a function tool
@function_tool
def compute_gpa(grades: list[float] | float) -> str:
    """
    Computes the GPA based on a single grade or a list of percentage grades.
    """
    # Ensure grades is a list
    if isinstance(grades, float) or isinstance(grades, int):
        grades = [grades]

    grade_points = [get_grade_point(grade) for grade in grades]
    print(grade_points)
    gpa = sum(grade_points) / len(grade_points)
    
    return f"The computed GPA is {gpa:.2f}"

In [9]:
# Create an GPA Agent to use compute_gpa fuction as a tool
gpa_agent = Agent(
    name="gpa_agent",
    instructions="You calculate GPA from percentage grades using tools only.",
    tools=[compute_gpa],
    model="gpt-4o"
)

In [10]:
result = await Runner.run(gpa_agent, input="Calculate GPA for grades 95, 88, and 72")
print(result.final_output)

[4.0, 3.67, 2.67]
The GPA for the grades 95, 88, and 72 is 3.45.


#### GPA Agent Workflow
| Step | What Happens                                     |
| ---- | ------------------------------------------------ |
| 1️⃣  | Agent receives string input                       |
| 2️⃣  | It checks available tool signatures               |
| 3️⃣  | Uses LLM to understand the user's instructions    |
| 4️⃣  | Parses string → structured input grades=[...]     |
| 5️⃣  | Calls your function with real Python data         |

In [11]:
# Define the prerequisite rules
FOUNDATION_COURSES = {"FPP", "MPP"}
PREREQUISITES = {
    "EA": {"DBMS", "WAP"},  # Either DBMS or WAP
    "MWA": {"WAP"},         # Requires WAP
}

In [12]:
@function_tool
def check_prerequisites(completed_courses: list[str], target_course: str) -> str:
    """
    Checks if the student has satisfied prerequisites for the target course.
    - FPP and MPP must be completed before taking any other course.
    - DBMS or WAP is required before EA.
    - WAP is required before MWA.
    """
    completed_set = set(course.upper() for course in completed_courses)
    target_course = target_course.upper()

    # Check mandatory foundation courses
    if not FOUNDATION_COURSES.issubset(completed_set):
        return "You must complete both FPP and MPP before attempting any other course."

    # Check if target course has any prerequisites
    if target_course in PREREQUISITES:
        required = PREREQUISITES[target_course]
        # For EA, check if at least one of DBMS or WAP is done
        if target_course == "EA":
            if not any(req in completed_set for req in required):
                return f"To take EA, you must complete at least one of: {', '.join(required)}."
        else:
            # All other courses require full prerequisite completion
            if not required.issubset(completed_set):
                return f"To take {target_course}, you must complete: {', '.join(required)}."

    return f"You are eligible to register for {target_course}."

In [ ]:
# Agent can use multiple tools[Builtintools, customtools]
support_agent = Agent(
    name="student_support_agent",
    instructions="You are a student advisor agent. Use tools to either calculate GPA or check course prerequisites based on the student's question.",
    tools=[compute_gpa, check_prerequisites],
    model="gpt-4o"
)

In [14]:
# GPA function call Example
result1 = await Runner.run(support_agent, input="Can you calculate my GPA from 95, 88, and 72?")
print(result1.final_output)



[4.0, 3.67, 2.67]
Your GPA is 3.45.


In [15]:
# Prerequisite function call Example
result2 = await Runner.run(support_agent, input="I completed FPP, MPP, and WAP. Can I take MWA?")
print(result2.final_output)

Yes, you are eligible to take MWA.


#### Factorial Agent 

In [1]:
# Factorial Tool
from agents.tool import function_tool

@function_tool
def factorial(n: int) -> int:
    """Returns the factorial of a number"""
    result = 1
    for i in range(1, n + 1):
        result *= i
    return result

In [4]:
fact_agent = Agent(
    name="FactorialAgent",
    instructions="You are an AI agent that calculates factorial using tools.",
    tools=[factorial]
)

In [6]:
result = await Runner.run(fact_agent, "What is the factorial of 5?")
print(result.final_output)


The factorial of 5 is 120.


#### External API call tool

APIs are also used to access real-time data, such as customer records, weather information, or cryptocurrency prices. In this example, create a tool that will fetch the current price of Bitcoin in USD. 

Python function, get_price_of_bitcoin() that calls the CoinGecko API (a free public API for cryptocurrency prices) via the requests library. When the agent calls the tool, the tool calls the API, and the information received from the API is then passed back to the agent.

response.json() -> converts the API response into a Python dictionary.

{

  "bitcoin": {

      "usd": 67450.23

  }

}

Syntax:
response.json()[key1][key2][key3]...



In [ ]:
# Create the tool
@function_tool
def get_price_of_bitcoin() -> str:
    """Get the price of Bitcoin."""
    url = "https://api.coingecko.com/api/v3/simple/price?ids=bitcoin&vs_currencies=usd"
    #Send an HTTP GET request
    response = requests.get(url)
    price = response.json()["bitcoin"]["usd"]
    return f"${price:,.2f} USD."

In [11]:
# Create the agent
crypto_agent = Agent(
    name="CryptoTracker",
    instructions="You are a crypto assistant. Use tools to get real-time data.",
    tools=[get_price_of_bitcoin]
)

In [12]:
# Run the agent with an example prompt
result = await Runner.run(crypto_agent, "What's the price of Bitcoin?")
print(result.final_output)


The current price of Bitcoin is $90,222.00 USD.
